# MLOps Training 2026/2027 — Task 1

## Get the Data Into a Database

This notebook loads the Olist Brazilian E-Commerce CSV files into a local PostgreSQL database and verifies the tables, schema, relationships, and delivery problem. Full EDA, feature engineering, and model training belong to later tasks.

Official dataset page: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


## Step 1 — Read and understand

The dataset is relational, not one ready-made ML table. Different files represent orders, customers, order items, payments, reviews, products, sellers, geolocation, and category translations. The principal keys are `order_id`, `customer_id`, `product_id`, and `seller_id`.

A future ML table must have one row per order. One-to-many tables must be aggregated by `order_id` before joining, otherwise numeric values can be duplicated. Reviews and actual delivery dates are useful for analysis but may cause leakage if used as features before delivery.


In [1]:
from pathlib import Path
import os
import pandas as pd

print(f"pandas version: {pd.__version__}")
archive_path = Path(r"C:\Users\HP\Desktop\MLOps\Task1\data")
if not archive_path.exists():
    raise FileNotFoundError(f"Dataset folder not found: {archive_path}")
print(f"Dataset folder: {archive_path}")


pandas version: 2.3.3
Dataset folder: C:\Users\HP\Desktop\MLOps\Task1\data


In [2]:
csv_files = sorted(archive_path.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found")
file_manifest = pd.DataFrame({"file_name": [p.name for p in csv_files], "size_mb": [round(p.stat().st_size / 1024**2, 2) for p in csv_files]})
print(f"Found {len(csv_files)} CSV files")
display(file_manifest)


Found 9 CSV files


,file_name,size_mb
0,olist_customers_dataset.csv,8.62
1,olist_geolocation_dataset.csv,58.44
2,olist_order_items_dataset.csv,14.72
3,olist_order_payments_dataset.csv,5.51
4,olist_order_reviews_dataset.csv,13.78
5,olist_orders_dataset.csv,16.84
6,olist_products_dataset.csv,2.27
7,olist_sellers_dataset.csv,0.17
8,product_category_name_translation.csv,0.00


## Step 2 — Connect to PostgreSQL

Connection settings are read from environment variables. The password is requested interactively when `OLIST_DB_PASSWORD` is not set, so credentials are not stored in the notebook.


In [3]:
import getpass
from sqlalchemy import create_engine, text

DB_HOST = os.getenv("OLIST_DB_HOST", "localhost")
DB_PORT = os.getenv("OLIST_DB_PORT", "5432")
DB_NAME = os.getenv("OLIST_DB_NAME", "olist")
DB_USER = os.getenv("OLIST_DB_USER", "postgres")
DB_PASSWORD = os.getenv("OLIST_DB_PASSWORD") or getpass.getpass("PostgreSQL password: ")
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}", pool_pre_ping=True)

with engine.connect() as connection:
    version = connection.execute(text("SELECT version();")).scalar_one()
print("PostgreSQL connection successful.")
print(version)


PostgreSQL connection successful.
PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit


## Step 2 — Load the CSV files

Each CSV is loaded into a table. The `_dataset.csv` suffix is removed from dataset filenames. Replacing tables is intentional for a reproducible local rebuild. Constraints can be applied afterward with `Relationships.sql`.


In [4]:
def table_name_from_file(file_name: str) -> str:
    suffix = "_dataset.csv" if file_name.endswith("_dataset.csv") else ".csv"
    return file_name[:-len(suffix)]

load_rows = []
for csv_path in csv_files:
    table_name = table_name_from_file(csv_path.name)
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10000)
    load_rows.append({"file_name": csv_path.name, "table_name": table_name, "rows_loaded": len(df), "columns": len(df.columns)})
load_manifest = pd.DataFrame(load_rows).sort_values("table_name").reset_index(drop=True)
display(load_manifest)
print("All CSV files loaded successfully.")


,file_name,table_name,rows_loaded,columns
0,olist_customers_dataset.csv,olist_customers,99441,5
1,olist_geolocation_dataset.csv,olist_geolocation,1000163,5
2,olist_order_items_dataset.csv,olist_order_items,112650,7
3,olist_order_payments_dataset.csv,olist_order_payments,103886,5
4,olist_order_reviews_dataset.csv,olist_order_reviews,99224,7
5,olist_orders_dataset.csv,olist_orders,99441,8
6,olist_products_dataset.csv,olist_products,32951,9
7,olist_sellers_dataset.csv,olist_sellers,3095,4
8,product_category_name_translation.csv,product_category_name_translation,71,2


All CSV files loaded successfully.


## Step 3 — Verify the database and schema


In [5]:
from sqlalchemy import inspect

expected_tables = {table_name_from_file(p.name) for p in csv_files}
inspector = inspect(engine)
actual_tables = set(inspector.get_table_names())
missing = expected_tables - actual_tables
if missing:
    raise AssertionError(f"Missing tables: {sorted(missing)}")

row_counts = []
schema_rows = []
with engine.connect() as connection:
    for table_name in sorted(expected_tables):
        count = connection.execute(text(f'SELECT COUNT(*) FROM "{table_name}"')).scalar_one()
        row_counts.append({"table_name": table_name, "row_count": count})
        for column in inspector.get_columns(table_name):
            schema_rows.append({"table_name": table_name, "column_name": column["name"], "data_type": str(column["type"])})
row_counts = pd.DataFrame(row_counts)
schema_df = pd.DataFrame(schema_rows)
display(row_counts)
print(f"Verified {len(row_counts)} tables and {row_counts.row_count.sum():,} total rows.")
display(schema_df)


,table_name,row_count
0,olist_customers,99441
1,olist_geolocation,1000163
2,olist_order_items,112650
3,olist_order_payments,103886
4,olist_order_reviews,99224
5,olist_orders,99441
6,olist_products,32951
7,olist_sellers,3095
8,product_category_name_translation,71


Verified 9 tables and 1,550,922 total rows.


,table_name,column_name,data_type
0,olist_customers,customer_id,TEXT
1,olist_customers,customer_unique_id,TEXT
2,olist_customers,customer_zip_code_prefix,BIGINT
3,olist_customers,customer_city,TEXT
4,olist_customers,customer_state,TEXT
5,olist_geolocation,geolocation_zip_code_prefix,BIGINT
6,olist_geolocation,geolocation_lat,DOUBLE PRECISION
7,olist_geolocation,geolocation_lng,DOUBLE PRECISION
8,olist_geolocation,geolocation_city,TEXT
9,olist_geolocation,geolocation_state,TEXT


## Step 3.1 — Test the key relationships

Each query tests one relationship independently. The output is limited only for readability.


In [6]:
join_queries = {
    "orders_customers": "SELECT o.order_id, o.customer_id, o.order_status, c.customer_city, c.customer_state FROM olist_orders o JOIN olist_customers c ON o.customer_id = c.customer_id LIMIT 5;",
    "orders_items_products": "SELECT o.order_id, oi.product_id, p.product_category_name, oi.price, oi.freight_value FROM olist_orders o JOIN olist_order_items oi ON o.order_id = oi.order_id JOIN olist_products p ON oi.product_id = p.product_id LIMIT 5;",
    "items_sellers": "SELECT oi.order_id, oi.order_item_id, oi.seller_id, s.seller_city, s.seller_state FROM olist_order_items oi JOIN olist_sellers s ON oi.seller_id = s.seller_id LIMIT 5;",
    "orders_payments": "SELECT o.order_id, op.payment_type, op.payment_value, op.payment_installments FROM olist_orders o JOIN olist_order_payments op ON o.order_id = op.order_id LIMIT 5;",
}
for name, query in join_queries.items():
    print(f"\n--- {name} ---")
    display(pd.read_sql_query(query, engine))



--- orders_customers ---


,order_id,customer_id,order_status,customer_city,customer_state
0,2711a938db643b3f0b62ee2c8a2784aa,29cb486c739f9774c8eb542e07b56cd2,delivered,brasilia,DF
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,sao paulo,SP
2,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,barreiras,BA
3,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,vianopolis,GO
4,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,sao goncalo do amarante,RN



--- orders_items_products ---


,order_id,product_id,product_category_name,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumaria,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,199.90,18.14



--- items_sellers ---


,order_id,order_item_id,seller_id,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,48436dade18ac8b2bce089ec2a041202,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,5b51032eddd242adc84c38acab88f23d,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,9d7a1d34a5052409006425275ba1c2b4,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,df560393f3a51e74553ab94004ba5c87,loanda,PR



--- orders_payments ---


,order_id,payment_type,payment_value,payment_installments
0,b81ef226f3fe1789b1e8b2acac839d17,credit_card,99.33,8
1,a9810da82917af2d9aefd1278f1dcfa0,credit_card,24.39,1
2,25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card,65.71,1
3,ba78997921bbcdc1373bb41e913ab953,credit_card,107.78,8
4,42fdf880ba16b47b59251dd489d4441a,credit_card,128.45,2


## Step 3.2 — Understand the delivery problem

For delivered orders with valid dates, `is_late` is 1 when the actual delivery date is after the estimated date, and 0 when it is on or before the estimate. The target is defined here for analysis; future features must represent information available at prediction time.


In [7]:
status_query = "SELECT order_status, COUNT(*) AS order_count, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage FROM olist_orders GROUP BY order_status ORDER BY order_count DESC;"
status_df = pd.read_sql_query(status_query, engine)
display(status_df)

late_query = "SELECT COUNT(*) AS delivered_orders, COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) AS late_orders, COUNT(*) FILTER (WHERE order_delivered_customer_date <= order_estimated_delivery_date) AS on_time_orders, ROUND(100.0 * COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date) / COUNT(*), 2) AS late_percentage FROM olist_orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL AND order_estimated_delivery_date IS NOT NULL;"
late_summary = pd.read_sql_query(late_query, engine)
display(late_summary)


,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


,delivered_orders,late_orders,on_time_orders,late_percentage
0,96470,7826,88644,8.11


## Optional database constraints

After ingestion and verification, `Relationships.sql` may be run to add primary and foreign keys. The category translation table is intentionally not linked by a foreign key because it does not cover every product category.


## Final completion check

Task 1 is complete when the database runs locally with data inside, the tables can be queried and joined, the relationships are understood, and the late-delivery problem is defined.


In [8]:
required = {"database_connection": False, "all_tables_present": False, "join_test_passed": False, "prediction_problem_defined": False}
with engine.connect() as connection:
    connection.execute(text("SELECT 1"))
required["database_connection"] = True
required["all_tables_present"] = expected_tables.issubset(set(inspect(engine).get_table_names()))
join_count = pd.read_sql_query("SELECT COUNT(*) AS joined_rows FROM olist_orders o JOIN olist_customers c ON o.customer_id = c.customer_id;", engine).iloc[0]["joined_rows"]
required["join_test_passed"] = join_count > 0
order_columns = set(schema_df.loc[schema_df.table_name == "olist_orders", "column_name"])
required["prediction_problem_defined"] = {"order_delivered_customer_date", "order_estimated_delivery_date"}.issubset(order_columns)
check_df = pd.DataFrame({"criterion": list(required), "passed": list(required.values())})
display(check_df)
assert all(required.values()), f"Completion check failed: {required}"
print("ALL TASK 1 COMPLETION CRITERIA PASSED")


,criterion,passed
0,database_connection,True
1,all_tables_present,True
2,join_test_passed,True
3,prediction_problem_defined,True


ALL TASK 1 COMPLETION CRITERIA PASSED
